# Nền tảng 8 — Kỹ thuật hiệu năng, qua đúng sự cố của project

Ngày chạy notebook 01 trên Kaggle, huấn luyện tokenizer **hết 12 giờ giới hạn phiên mà chưa xong**. Giai đoạn 1
(BPE thường) chỉ mất **khoảng 2 phút**; giai đoạn 2 của SuperBPE thì không kết thúc.
(Số 2 phút đọc từ `kaggle/outputs/vitok-data/notebook01.log`: lệnh huấn luyện 16k bắt đầu ở giây 613,
cả NFC hoá corpus lẫn giai đoạn 1 xong trước giây 722.)

Notebook này đi lại toàn bộ đường từ triệu chứng tới bản vá, vì đó là một ví dụ trọn vẹn của ba kỹ năng hiệu
năng dùng được ở mọi nơi khác:

1. **Đo trước khi đoán** — tìm đúng đại lượng đã nổ, thay vì tối ưu chỗ dễ nhìn.
2. **Phân tích độ phức tạp** — biết đại lượng nào nhân với đại lượng nào.
3. **Vector hoá** — chuyển vòng lặp Python thành thao tác trên toàn mảng numpy.

Kết quả cuối: giai đoạn 2 trên 500MB chạy trong **182 giây encode + 238 giây gộp**, thay vì không xong trong 12
giờ.

**Chạy bằng kernel pixi của project.**

In [ ]:
import cProfile
import io
import json
import math
import os
import pstats
import sys
import time
import unicodedata
from collections import Counter
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
while not (ROOT / "pixi.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
DATA = ROOT / "kaggle" / "outputs" / "vitok-data"
WORK = Path(os.environ.get("TMPDIR", "/tmp")) / "vitok_nb08"
WORK.mkdir(exist_ok=True)

from vitok import superbpe
from vitok.tokenizer_spec import STAGE1_REGEX, STAGE2_REGEX
from vitok.train_tokenizers import _pre_tokenizer

docs = [json.loads(l)["text"] for l in (DATA / "test.jsonl").open(encoding="utf-8")]
mau = unicodedata.normalize("NFC", "\n".join(docs[:300]))
mau_path = WORK / "mau.txt"
mau_path.write_text(mau, encoding="utf-8")
print(f"corpus mẫu: {len(mau):,} ký tự ({len(mau.encode()) / 1e6:.2f} MB)")

## 1. Triệu chứng và giả thuyết đầu tiên

Log dừng ở dòng tiến độ của `BpeTrainer`, không có lỗi. Giả thuyết đầu tiên — và là giả thuyết sai — là "corpus
500MB quá lớn". Nó sai vì **giai đoạn 1 chạy trên đúng corpus đó và xong trong khoảng 2 phút**. Hai giai đoạn khác nhau
đúng một thứ: **biểu thức chính quy cắt pretoken**.

Nên câu hỏi đúng không phải "dữ liệu có lớn không" mà là **"đổi regex làm đại lượng nào thay đổi?"**

Nhắc lại cách trainer của `tokenizers` làm việc (notebook 04 mục 3–4):

1. Cắt văn bản thành pretoken bằng regex.
2. **Gom pretoken trùng nhau** thành `(pretoken, số lần)` — đây là bước nén làm mọi thứ chạy được.
3. Lặp: đếm mọi cặp liền nhau (có trọng số theo số lần), gộp cặp nhiều nhất, cập nhật các pretoken.

Bước 2 là chỗ giả định ngầm nằm: nó chỉ có ích khi pretoken **ngắn và lặp lại nhiều**.

In [ ]:
def thong_ke_pretoken(regex, text):
    toks = [t for t, _ in _pre_tokenizer(regex).pre_tokenize_str(text)]
    dem = Counter(toks)
    L = np.array([len(t) for t in toks])
    L_uniq = np.array([len(t) for t in dem])
    return {
        "số pretoken": len(toks),
        "pretoken khác nhau": len(dem),
        "tỷ lệ trùng lặp": 1 - len(dem) / len(toks),
        "độ dài trung bình": L.mean(),
        "độ dài trung vị": float(np.median(L)),
        "độ dài p99": float(np.percentile(L, 99)),
        "độ dài lớn nhất": int(L.max()),
        "Σ L (công việc tuyến tính)": int(L_uniq.sum()),
        "Σ L² (công việc bậc hai)": int((L_uniq.astype(np.int64) ** 2).sum()),
    }


s1 = thong_ke_pretoken(STAGE1_REGEX, mau)
s2 = thong_ke_pretoken(STAGE2_REGEX, mau)
print(f" {'đại lượng':30s} | {'giai đoạn 1':>16s} | {'giai đoạn 2':>16s} | tỷ lệ")
for k in s1:
    a, b = s1[k], s2[k]
    ty = f"{b / a:,.1f}×" if a else "-"
    fa = f"{a:,.1f}" if isinstance(a, float) else f"{a:,}"
    fb = f"{b:,.1f}" if isinstance(b, float) else f"{b:,}"
    print(f" {k:30s} | {fa:>16s} | {fb:>16s} | {ty:>10s}")

Bảng trên là toàn bộ chẩn đoán, đọc từ dưới lên:

- **Độ dài trung bình** nhảy từ ~5 ký tự lên hơn 100. Lý do: regex giai đoạn 2 **không tách theo dấu cách đơn**,
  nên một pretoken có thể kéo dài qua cả đoạn văn, chỉ bị cắt ở số, dấu câu nhiều ký tự và xuống dòng. Pretoken
  dài nhất trong mẫu này vượt 4.000 ký tự.
- **Tỷ lệ trùng lặp** sụp từ ~94% xuống ~58%. Một câu dài gần như không bao giờ lặp lại nguyên văn, nên bước gom
  `(pretoken, số lần)` hầu như không nén được gì nữa. Giả định ngầm của trainer bị phá.
- **$\sum L^2$** tăng hàng nghìn lần, trong khi $\sum L$ gần như không đổi (nó luôn xấp xỉ kích thước corpus).

Vì sao $\sum L^2$ mới là đại lượng quan trọng: cài đặt `Word::merge` của `tokenizers` duyệt lại **toàn bộ**
pretoken mỗi lần gộp một cặp bên trong nó. Một pretoken dài $L$ phải chịu khoảng $L$ lần gộp trong suốt quá
trình, mỗi lần tốn $O(L)$, nên tổng công việc trên pretoken đó là $O(L^2)$ (ký hiệu $O$: xem notebook 04). Tổng lại trên cả corpus:

$$W \approx \sum_{\text{pretoken duy nhất}} L^2$$

**Ký hiệu mới:** $W$ — tổng công việc của trainer; $L$ — độ dài một pretoken (không phải số lớp); tổng lấy qua
các pretoken **khác nhau**.

Với $L \approx 5$ thì $L^2 = 25$; với $L \approx 113$ thì $L^2 \approx 12\,800$. Trên corpus thật, chi phí đó
nhân với hàng trăm triệu.

In [ ]:
ty_le_mau = s2["Σ L² (công việc bậc hai)"] / s1["Σ L² (công việc bậc hai)"]
byte_mau = len(mau.encode())
byte_that = 500_181_599                                   # corpus thật, xem train_meta.json

print(f"trên mẫu {byte_mau / 1e6:.2f} MB: Σ L² của giai đoạn 2 gấp {ty_le_mau:,.0f} lần giai đoạn 1")
print(f"\nΣ L² tăng theo cỡ corpus như thế nào?")
for phan in (0.25, 0.5, 1.0):
    con = mau[: int(len(mau) * phan)]
    t2 = thong_ke_pretoken(STAGE2_REGEX, con)
    print(f"  {phan:4.0%} corpus: Σ L² = {t2['Σ L² (công việc bậc hai)']:>15,}"
          f" | dài trung bình {t2['độ dài trung bình']:6.1f}")
print("\n=> Σ L² tăng gần tuyến tính theo cỡ corpus, nhưng với HỆ SỐ khổng lồ so với giai đoạn 1.")
print(f"   Ngoại suy sang corpus thật ({byte_that / 1e6:.0f} MB) thì công việc lớn hơn"
      f" khoảng {byte_that / byte_mau:.0f} lần con số trên.")

Còn một tầng chi phí thứ hai trong cài đặt gốc: sau mỗi lần gộp, bảng đếm cặp được **dựng lại** từ tập các
pretoken bị ảnh hưởng, chứ không cập nhật theo sai khác. Ở giai đoạn 1 thì rẻ, vì mỗi cặp chỉ xuất hiện trong ít
pretoken ngắn. Ở giai đoạn 2, một cặp phổ biến nằm trong hàng triệu pretoken dài, nên mỗi lần gộp kéo theo một
lượt quét gần như toàn bộ dữ liệu.

Hai tầng nhân nhau: $1\,600$ merge × (quét lại dữ liệu + công việc $O(L^2)$). Đó là lý do nó không xong trong 12
giờ, và cũng là lý do **không thể sửa bằng cách chờ lâu hơn hay thuê máy to hơn** — phải đổi thuật toán.

## 2. Thiết kế lại: danh sách liên kết trên mảng phẳng

Ba quyết định của `vitok/superbpe.py`:

**(a) Một mảng phẳng thay vì danh sách các pretoken.** Toàn corpus sau khi áp merge kế thừa được nối thành **một**
mảng `uint32`, các pretoken ngăn nhau bằng sentinel `SEP`. Không còn vòng lặp Python trên từng pretoken.

**(b) Danh sách liên kết bằng hai mảng chỉ số** `nxt` và `prv`. Khi gộp hai vị trí $i, j$, ta ghi token mới vào
$i$, đánh dấu $j$ là `HOLE`, rồi **nối lại con trỏ**: `nxt[i] = nxt[j]`. Không phải dịch chuyển phần tử, nên một
lần gộp tốn $O(\text{số lần cặp đó xuất hiện})$ chứ không phải $O(L)$ cho mỗi pretoken chứa nó.

**(c) Đếm cặp theo sai khác.** Sau khi gộp, chỉ các cặp **liền kề chỗ vừa đổi** thay đổi số lượng. Thay vì dựng
lại bảng đếm, ta trừ các cặp cũ và cộng các cặp mới:

$$\text{counts} \mathrel{{-}{=}} \{(\text{prv}_i, a), (a, b), (b, \text{nxt}_j)\}, \qquad \text{counts} \mathrel{{+}{=}} \{(\text{prv}_i, \text{new}), (\text{new}, \text{nxt}_j)\}$$

**Ký hiệu mới**
- $a$, $b$ — cặp vừa gộp; $\text{new}$ — id token mới $ab$; $i$, $j$ — hai vị trí đang chứa $a$ và $b$
- $\text{prv}_i$, $\text{nxt}_j$ — token ngay trước $i$ và ngay sau $j$

Tổng công việc cho một merge trở thành $O(\text{số lần xuất hiện của cặp đó})$, và tổng trên toàn bộ quá trình
là $O(\sum L)$ chứ không phải $O(\sum L^2)$.

Một chi tiết phải xử lý riêng: khi $a = b$ (ví dụ "a a a a"), các cặp **chồng lấn nhau** và không được gộp tất cả
cùng lúc — phải lấy xen kẽ, đúng như `Word::merge` làm khi duyệt từ trái sang phải. Đoạn mã lo việc đó:

```python
if a == b and len(i) > 1:
    linked = np.concatenate(([False], i[1:] == j[:-1]))
    chain_start = np.flatnonzero(~linked)
    offset = np.arange(len(i)) - chain_start[np.cumsum(~linked) - 1]
    i, j = i[offset % 2 == 0], j[offset % 2 == 0]
```

Cell dưới minh hoạ đúng luật đó trên một chuỗi nhỏ.

In [ ]:
def gop_trai_sang_phai(day, a, b, moi):
    """Luật của Word::merge: duyệt trái sang phải, token vừa tạo không được ghép tiếp ngay."""
    ra, i = [], 0
    while i < len(day):
        if i + 1 < len(day) and day[i] == a and day[i + 1] == b:
            ra.append(moi)
            i += 2
        else:
            ra.append(day[i])
            i += 1
    return ra


day = list("aaaaa")
print(f"chuỗi {day}, gộp cặp ('a','a') -> 'A'")
print(f"  kết quả đúng: {gop_trai_sang_phai(day, 'a', 'a', 'A')}")
print(f"  nếu gộp tất cả vị trí khớp cùng lúc: ['A','A','A','A'] — sai, vì các cặp chồng lấn nhau")

tok = np.array([superbpe.SEP if c == "|" else ord(c) for c in "aaaaa|abab"], dtype=np.int64)
print(f"\nmảng phẳng với sentinel SEP ngăn pretoken: {['SEP' if t == superbpe.SEP else chr(t) for t in tok]}")
print(f"SEP = {superbpe.SEP} và HOLE = {superbpe.HOLE} nằm ngoài vùng id thật, nên không bao giờ lẫn với token.")

## 3. Kiểm chứng: bản numpy có cho ra **đúng** kết quả cũ không?

Tối ưu mà đổi kết quả thì vô nghĩa. Bản tham chiếu dưới đây là bản chậm, viết thẳng theo mô tả của
`do_train_extend` trong fork SuperBPE: gộp từng pretoken một, và **đếm lại toàn bộ cặp sau mỗi bước**. Nó cũng
là bản đang dùng trong `tests/test_superbpe.py`.

Hai bản phải ra cùng một danh sách merge, kể cả quy tắc phá hoà (ưu tiên cặp có id nhỏ hơn) và các luật cấm
(`:Ġ`, tối đa 4 từ).

In [ ]:
def tham_chieu_stage2(lines, vocab, inherited, vocab_size, max_words=4):
    """Bản chậm, viết thẳng theo mô tả: gộp từng pretoken, đếm lại mọi cặp sau mỗi bước."""
    pre = _pre_tokenizer(STAGE2_REGEX)
    vocab = dict(vocab)
    id_to_token = {i: t for t, i in vocab.items()}
    words = [[vocab[c] for c in piece] for line in lines for piece, _ in pre.pre_tokenize_str(line)]
    for a, b in inherited:
        words = [gop_trai_sang_phai(w, vocab[a], vocab[b], vocab[a + b]) for w in words]

    merges, cam = [], set()
    while len(vocab) < vocab_size:
        dem = Counter(p for w in words for p in zip(w, w[1:]))
        dem = {p: c for p, c in dem.items() if p not in cam}
        if not dem:
            break
        cap = min(dem, key=lambda p: (-dem[p], p))                  # nhiều nhất; hoà thì id nhỏ hơn
        token = id_to_token[cap[0]] + id_to_token[cap[1]]
        if ":Ġ" in token or len([x for x in token.split("Ġ") if x]) > max_words:
            cam.add(cap)
            continue
        moi = vocab.setdefault(token, len(vocab))
        id_to_token[moi] = token
        merges.append((id_to_token[cap[0]], id_to_token[cap[1]]))
        words = [gop_trai_sang_phai(w, *cap, moi) for w in words]
    return vocab, merges


# corpus rất nhỏ, vì bản tham chiếu chậm
nho = "\n".join(docs[:12])
nho_path = WORK / "nho.txt"
nho_path.write_text(unicodedata.normalize("NFC", nho), encoding="utf-8")

from tokenizers import Tokenizer, models, pre_tokenizers, trainers

t1 = Tokenizer(models.BPE())
t1.pre_tokenizer = _pre_tokenizer(STAGE1_REGEX)
t1.train([str(nho_path)], trainers.BpeTrainer(vocab_size=600, show_progress=False,
                                              initial_alphabet=pre_tokenizers.ByteLevel.alphabet()))
cwd = os.getcwd()
os.chdir(WORK)
t1.model.save(".")
os.chdir(cwd)
merges_txt = (WORK / "merges.txt").read_text(encoding="utf-8").splitlines()[1:]
vocab_txt = json.loads((WORK / "vocab.json").read_text(encoding="utf-8"))
n_alpha = len(vocab_txt) - len(merges_txt)
n_ke_thua = round(0.9 * 600) - n_alpha
ke_thua = [tuple(m.split(" ")) for m in merges_txt[:n_ke_thua]]
v_ke_thua = {t: i for t, i in vocab_txt.items() if i < n_alpha + len(ke_thua)}
print(f"giai đoạn 1: {len(merges_txt)} merge | kế thừa {len(ke_thua)} | vocab đích 600")

In [ ]:
lines = nho_path.read_text(encoding="utf-8").splitlines(keepends=True)

t0 = time.time()
v_ref, m_ref = tham_chieu_stage2(lines, v_ke_thua, ke_thua, 600)
t_ref = time.time() - t0

t0 = time.time()
ids = superbpe.encode_corpus([str(nho_path)], v_ke_thua, ke_thua, _pre_tokenizer(STAGE2_REGEX))
v_fast, m_fast = superbpe.train_stage2(ids, v_ke_thua, 600, log_every=10 ** 9)
t_fast = time.time() - t0

print(f"bản tham chiếu: {len(m_ref):3d} merge trong {t_ref:6.2f}s")
print(f"bản numpy     : {len(m_fast):3d} merge trong {t_fast:6.2f}s  ({t_ref / t_fast:.1f}× nhanh hơn)")
print(f"\ndanh sách merge GIỐNG HỆT nhau: {m_ref == m_fast}")
print(f"vocab giống hệt nhau          : {v_ref == v_fast}")
print(f"\n5 merge đầu của cả hai: {m_fast[:5]}")

## 4. Tốc độ tăng theo cỡ dữ liệu như thế nào

Con số "nhanh hơn bao nhiêu lần" trên một corpus bé không nói được nhiều; điều cần biết là **độ dốc**: khi dữ
liệu lớn gấp đôi thì thời gian tăng bao nhiêu.

Điều sẽ thấy trong bảng: bản tham chiếu tăng xấp xỉ gấp đôi mỗi lần dữ liệu gấp đôi (nó phải quét lại toàn bộ
pretoken sau mỗi merge), còn bản numpy gần như **đứng yên** — ở cỡ này thời gian của nó bị chi phối bởi chi phí
cố định của vài trăm lời gọi numpy, chứ chưa phải bởi dữ liệu. Vì vậy khoảng cách giữa hai bản **giãn ra** theo
cỡ corpus: 15× ở 9KB, 70× ở 84KB, và trên 500MB thì là "không xong trong 12 giờ" so với 7 phút.

In [ ]:
print(" số văn bản | byte      | tham chiếu | numpy    | tỷ lệ")
truoc = None
for n_doc in (4, 8, 16, 32):
    txt = unicodedata.normalize("NFC", "\n".join(docs[:n_doc]))
    p = WORK / f"c{n_doc}.txt"
    p.write_text(txt, encoding="utf-8")
    lines = p.read_text(encoding="utf-8").splitlines(keepends=True)

    t0 = time.time()
    tham_chieu_stage2(lines, v_ke_thua, ke_thua, 600)
    t_r = time.time() - t0

    t0 = time.time()
    i2 = superbpe.encode_corpus([str(p)], v_ke_thua, ke_thua, _pre_tokenizer(STAGE2_REGEX))
    superbpe.train_stage2(i2, v_ke_thua, 600, log_every=10 ** 9)
    t_f = time.time() - t0

    ghi = ""
    if truoc:
        ghi = f"  (dữ liệu ×{len(txt.encode()) / truoc[0]:.1f} -> tham chiếu ×{t_r / truoc[1]:.1f}, numpy ×{t_f / truoc[2]:.1f})"
    truoc = (len(txt.encode()), t_r, t_f)
    print(f" {n_doc:10d} | {len(txt.encode()):9,} | {t_r:9.2f}s | {t_f:7.2f}s | {t_r / t_f:6.1f}×{ghi}")

## 5. Thời gian đi đâu: đọc profile

Khi đã vector hoá, câu hỏi tiếp theo là phần nào còn lại là nút thắt. `cProfile` của thư viện chuẩn đủ dùng: nó
đếm thời gian theo hàm. Với mã numpy, điều cần nhìn là **số lần gọi**: nếu một hàm numpy được gọi hàng triệu lần
trên mảng bé thì chi phí gọi hàm Python áp đảo phần tính toán, và đó là dấu hiệu còn vòng lặp cần vector hoá.

In [ ]:
p = WORK / "c32.txt"
ids32 = superbpe.encode_corpus([str(p)], v_ke_thua, ke_thua, _pre_tokenizer(STAGE2_REGEX))

pr = cProfile.Profile()
pr.enable()
superbpe.train_stage2(ids32, v_ke_thua, 600, log_every=10 ** 9)
pr.disable()

buf = io.StringIO()
pstats.Stats(pr, stream=buf).sort_stats("cumulative").print_stats(12)
print("\n".join(buf.getvalue().splitlines()[4:20]))

## 6. Vì sao vector hoá nhanh hơn đến thế

Một vòng lặp Python trên mảng phải, với **mỗi phần tử**: lấy đối tượng, kiểm tra kiểu, gọi hàm, cấp phát kết quả.
Một thao tác numpy làm cùng công việc đó bằng một vòng lặp C trên bộ nhớ liền kề, không kiểm tra kiểu và tận dụng
lệnh SIMD.

Chênh lệch thực tế thường là 20–100 lần, và cell dưới đo trực tiếp. Đây cũng là lý do `_merge` trong
`vitok/superbpe.py` không có vòng `for` nào chạy theo số lần xuất hiện của cặp: tất cả nằm trong `flatnonzero`,
`bincount`, `unique` — các hàm chạy trên toàn mảng.

In [ ]:
x = np.random.randint(0, 1000, size=2_000_000, dtype=np.int64)

t0 = time.time()
ra_py = [v + 1 for v in x.tolist()]
t_py = time.time() - t0

t0 = time.time()
ra_np = x + 1
t_np = time.time() - t0
print(f"cộng 1 vào 2 triệu phần tử: Python {t_py:.3f}s | numpy {t_np:.4f}s | {t_py / t_np:.0f}× nhanh hơn")

t0 = time.time()
dem_py = Counter(x.tolist())
t_py = time.time() - t0
t0 = time.time()
dem_np = np.bincount(x)
t_np = time.time() - t0
print(f"đếm tần suất              : Counter {t_py:.3f}s | bincount {t_np:.4f}s | {t_py / t_np:.0f}× nhanh hơn")

t0 = time.time()
vt_py = [i for i, v in enumerate(x.tolist()) if v == 7]
t_py = time.time() - t0
t0 = time.time()
vt_np = np.flatnonzero(x == 7)
t_np = time.time() - t0
print(f"tìm mọi vị trí bằng 7     : Python {t_py:.3f}s | flatnonzero {t_np:.4f}s | {t_py / t_np:.0f}× nhanh hơn")

## 7. Bộ nhớ và kiểu dữ liệu

Corpus thật sinh ra **104.564.258** id ở giai đoạn 2 (`train_meta.json`). Với mỗi id cần thêm hai con trỏ `nxt`
và `prv`, kiểu dữ liệu quyết định trực tiếp việc chạy được hay OOM trên máy Kaggle 30GB.

In [ ]:
N = 104_564_258
meta = json.loads((DATA / "tokenizers-16k" / "train_meta.json").read_text())
print("số id thật ở giai đoạn 2:", f"{meta['nfc']['stage2_ids']:,}")
print("\nba mảng (tok, nxt, prv) với các lựa chọn kiểu dữ liệu:\n")
print(" kiểu     | byte/phần tử | tổng cho 3 mảng | ghi chú")
for kieu, byte, ghi in (("uint16", 2, "quá nhỏ: vocab 16k vừa, nhưng SEP/HOLE thì không"),
                        ("uint32", 4, "đủ cho vocab tới 4 tỷ — lựa chọn của tok"),
                        ("int64", 8, "mặc định của numpy; chỉ số cần kiểu này")):
    print(f" {kieu:8s} | {byte:12d} | {3 * N * byte / 1e9:13.1f} GB | {ghi}")

print(f"\nthực tế trong vitok.superbpe: tok là uint32 ({N * 4 / 1e9:.1f} GB),")
print(f"còn nxt/prv là int64 vì chúng là CHỈ SỐ và cần giá trị âm (-1) để đánh dấu hết chuỗi:"
      f" {2 * N * 8 / 1e9:.1f} GB")
print(f"tổng ≈ {(N * 4 + 2 * N * 8) / 1e9:.1f} GB, vừa với RAM 30GB của Kaggle cùng các mảng tạm.")

print("\nthời gian thật đo được trên corpus 500MB (train_meta.json):")
for norm in ("nfc", "nfd"):
    m = meta[norm]
    print(f"  {norm}: encode {m['encode_seconds']}s + gộp {m['merge_seconds']}s"
          f" = {(m['encode_seconds'] + m['merge_seconds']) / 60:.1f} phút cho {m['stage2_new_merges']} merge mới")

## 8. Bài học rút ra

| Bài học | Biểu hiện trong sự cố này |
|---|---|
| Đo, đừng đoán | "Corpus quá lớn" là sai; cùng corpus đó giai đoạn 1 chạy xong trong ~2 phút |
| Tìm đại lượng đã nổ | Không phải $n$, mà là $\sum L^2$ và tỷ lệ trùng lặp của pretoken |
| Kiểm tra giả định ngầm của thư viện | Trainer giả định pretoken ngắn và lặp nhiều; regex mới phá cả hai |
| Đổi thuật toán trước khi đổi phần cứng | $O(\sum L^2) \to O(\sum L)$ không thể bù bằng máy to hơn |
| Giữ một bản tham chiếu chậm | Là cách duy nhất chứng minh bản nhanh không đổi kết quả |
| Vector hoá = bỏ vòng lặp Python | 20–100 lần, đo được ở mục 6 |
| Kiểu dữ liệu là quyết định bộ nhớ | uint32 so với int64 chênh 1,3 GB trên một mảng |

## 9. Câu hỏi tự kiểm

1. Vì sao "corpus 500MB quá lớn" là chẩn đoán sai?
2. Regex giai đoạn 2 làm đại lượng nào tăng, và vì sao $\sum L$ gần như không đổi?
3. Vì sao bước gom `(pretoken, số lần)` mất tác dụng ở giai đoạn 2?
4. Danh sách liên kết tránh được chi phí nào so với việc dựng lại pretoken sau mỗi lần gộp?
5. Đếm cặp theo sai khác cần biết những cặp nào thay đổi. Liệt kê chúng cho một lần gộp tại vị trí $i, j$.
6. Vì sao trường hợp $a = b$ phải xử lý riêng?
7. Nếu bản numpy nhanh hơn nhưng ra danh sách merge khác một chút, có dùng được không? Vì sao?
8. Nhìn profile ở mục 5: dấu hiệu nào cho biết còn một vòng lặp Python chưa vector hoá?

**Đáp án gợi ý**

1. Vì giai đoạn 1 chạy trên cùng corpus và xong trong khoảng 2 phút; biến duy nhất thay đổi là regex.
2. $\sum L^2$ tăng hàng nghìn lần. $\sum L$ luôn xấp xỉ tổng số ký tự corpus, không phụ thuộc cách cắt.
3. Vì pretoken dài gần như không lặp lại nguyên văn: tỷ lệ trùng lặp rơi từ ~94% xuống ~58%.
4. Chi phí $O(L)$ cho mỗi lần gộp bên trong một pretoken dài; nối con trỏ chỉ tốn $O(1)$ mỗi vị trí.
5. Trừ $(\mathrm{prv}_i, a)$, $(a, b)$, $(b, \mathrm{nxt}_j)$; cộng $(\mathrm{prv}_i, \text{new})$,
   $(\text{new}, \mathrm{nxt}_j)$.
6. Vì các cặp chồng lấn nhau: trong "aaaa" không thể gộp cả ba vị trí; phải lấy xen kẽ đúng như duyệt trái sang phải.
7. Không, trừ khi chứng minh được khác biệt không ảnh hưởng kết quả khoa học — tokenizer khác nghĩa là toàn bộ
   so sánh H1 đo trên hai vật khác nhau.
8. Một hàm có số lần gọi rất lớn (hàng triệu) nhưng thời gian mỗi lần gọi rất nhỏ — chi phí gọi hàm Python áp đảo.

**Nguồn đọc thêm**

- `src/vitok/superbpe.py` — bản cài đặt đầy đủ, khoảng 200 dòng.
- `tests/test_superbpe.py` — bản tham chiếu và các ca kiểm thử cho luật chồng lấn.
- [NumPy: broadcasting và vectorization](https://numpy.org/doc/stable/user/basics.broadcasting.html).
- [Python Profilers](https://docs.python.org/3/library/profile.html) — cProfile và pstats.